# Previsão da classificação final da época regular

Este notebook treina um modelo Ridge para prever o `rank` das equipas NHL. Usa as estatísticas agregadas dos jogadores nas até três épocas anteriores da mesma franquia. As posições previstas na liga e na divisão são calculadas separadamente por época. A validação é cronológica: as épocas mais recentes ficam reservadas para teste.

Antes de executar, gere os dados curados com `curate_player_stats.py` (ou execute a célula correspondente no notebook de análise). O modelo treinado fica na variável `final_model`; não é criado nenhum ficheiro `.pkl`.

In [106]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path.cwd()
if not (DATA_DIR / 'Teams.csv').exists():
    DATA_DIR = DATA_DIR / 'hockey_development_data'

STANDINGS_SOURCE = DATA_DIR / 'Teams.csv'
TEAM_STATS_SOURCE = DATA_DIR / 'team_stats_curated.csv'
LOOKBACK_SEASONS = 3
RIDGE_ALPHA = 1.0

TEAM_FEATURES = [
    'Forward_SER_Avg',
    'Defense_SER_Avg',
    'Goalie_GSAx_PG',
    'Team_PDO',
    'Experience_Factor',
    'Age_Index',
]
FEATURE_COLUMNS = [
    f'{feature}_lag_{lag}'
    for lag in range(1, LOOKBACK_SEASONS + 1)
    for feature in TEAM_FEATURES
]

print(f'Dados: {DATA_DIR.resolve()}')

Dados: C:\Users\cosme\UNI\AC\AC-ML-project\hockey_development_data


## Preparar exemplos de treino

O `rank` da época vem de `Teams.csv`. As características vêm de `team_stats_curated.csv` e são ligadas à franquia por época. Para cada equipa, só são consideradas épocas anteriores à época prevista. A previsão é ordenada dentro da liga e, separadamente, dentro de cada divisão. Épocas sem divisão registada terão `rank_previsto_divisao` vazio.

In [107]:
standings = pd.read_csv(STANDINGS_SOURCE, low_memory=False)
standings.columns = standings.columns.str.strip()
standings['lgID'] = standings['lgID'].astype('string').str.strip()
standings = standings.loc[standings['lgID'].eq('NHL')].copy()

for column in ['year', 'rank']:
    standings[column] = pd.to_numeric(standings[column], errors='coerce')
standings['tmID'] = standings['tmID'].astype('string').str.strip()
standings['divID'] = (
    standings['divID'].astype('string').str.strip().replace('', pd.NA)
)
standings['franchID'] = (
    standings['franchID'].astype('string').str.strip().replace('', pd.NA)
)
standings = standings.dropna(subset=['year', 'rank', 'tmID', 'franchID']).copy()
standings['year'] = standings['year'].astype(int)

team_stats = pd.read_csv(TEAM_STATS_SOURCE, low_memory=False)
team_stats.columns = team_stats.columns.str.strip()
missing_features = set(TEAM_FEATURES).difference(team_stats.columns)
if missing_features:
    raise KeyError(
        'Faltam métricas em team_stats_curated.csv: '
        + ', '.join(sorted(missing_features))
    )

team_stats['tmID'] = team_stats['tmID'].astype('string').str.strip()
team_stats['year'] = pd.to_numeric(team_stats['year'], errors='coerce')
team_stats = team_stats.dropna(subset=['year', 'tmID']).copy()
team_stats['year'] = team_stats['year'].astype(int)
team_stats[TEAM_FEATURES] = team_stats[TEAM_FEATURES].apply(
    pd.to_numeric, errors='coerce'
)

franchise_by_season = standings[['tmID', 'year', 'franchID']].drop_duplicates()
history = team_stats.merge(
    franchise_by_season,
    on=['tmID', 'year'],
    how='inner',
    validate='one_to_one',
)
history = (
    history.groupby(['franchID', 'year'], as_index=False)[TEAM_FEATURES]
    .mean()
    .sort_values(['franchID', 'year'])
)
history_by_franchise = {
    franchise: seasons.set_index('year')[TEAM_FEATURES].sort_index()
    for franchise, seasons in history.groupby('franchID', sort=False)
}

rows = []
for team in standings.itertuples(index=False):
    franchise_history = history_by_franchise.get(team.franchID)
    if franchise_history is None:
        continue

    previous = franchise_history.loc[
        franchise_history.index < team.year
    ].sort_index(ascending=False).head(LOOKBACK_SEASONS)
    if previous.empty:
        continue

    row = {
        'year': team.year,
        'lgID': team.lgID,
        'divID': team.divID,
        'tmID': team.tmID,
        'franchID': team.franchID,
        'rank': float(team.rank),
    }
    for lag in range(1, LOOKBACK_SEASONS + 1):
        season_stats = previous.iloc[lag - 1] if lag <= len(previous) else None
        for feature in TEAM_FEATURES:
            row[f'{feature}_lag_{lag}'] = (
                season_stats[feature] if season_stats is not None else np.nan
            )
    rows.append(row)

if not rows:
    raise ValueError(
        'Não foram criados exemplos. Execute primeiro curate_player_stats.py.'
    )

training_data = (
    pd.DataFrame(rows)
    .sort_values(['year', 'tmID'])
    .reset_index(drop=True)
)
print(f'Épocas: {training_data.year.min()}–{training_data.year.max()}')
print(f'Exemplos: {len(training_data):,}; métricas: {len(FEATURE_COLUMNS)}')
training_data[['year', 'lgID', 'divID', 'tmID', 'franchID', 'rank']].head()

Épocas: 1918–2009
Exemplos: 1,229; métricas: 18


,year,lgID,divID,tmID,franchID,rank
0,1918,NHL,NaN,MTL,MTL,2.0
1,1918,NHL,NaN,OTS,STE,1.0
2,1918,NHL,NaN,TOA,TOR,3.0
3,1919,NHL,NaN,MTL,MTL,2.0
4,1919,NHL,NaN,OTS,STE,1.0


## Definir e treinar o modelo Ridge

Os valores ausentes são preenchidos com a mediana calculada apenas nos dados de treino. As características são normalizadas antes do ajuste.

In [108]:
def fit_ridge(features, target, alpha=RIDGE_ALPHA):
    values = np.array(features, dtype=np.float64, copy=True)
    values[~np.isfinite(values)] = np.nan

    medians = np.array([
        np.median(column[np.isfinite(column)])
        if np.isfinite(column).any()
        else 0.0
        for column in values.T
    ])
    values = np.where(np.isfinite(values), values, medians)
    means = values.mean(axis=0)
    scales = values.std(axis=0)
    scales[scales == 0] = 1.0

    scaled = (values - means) / scales
    target_mean = float(np.mean(target))
    coefficients = np.linalg.solve(
        scaled.T @ scaled + alpha * np.eye(scaled.shape[1]),
        scaled.T @ (target - target_mean),
    )
    return {
        'medians': medians,
        'means': means,
        'scales': scales,
        'coefficients': coefficients,
        'intercept': target_mean,
    }


def predict_ridge(model, features):
    values = np.array(features, dtype=np.float64, copy=True)
    values[~np.isfinite(values)] = np.nan
    values = np.where(np.isfinite(values), values, model['medians'])
    scaled = (values - model['means']) / model['scales']
    return scaled @ model['coefficients'] + model['intercept']

## Avaliação cronológica usando todas as épocas

A avaliação walk-forward percorre todas as épocas com histórico anterior disponível: para cada época, treina apenas com épocas anteriores e prevê as equipas dessa época. Assim, cada época elegível participa na avaliação sem usar os seus resultados no treino. Depois, o modelo final é treinado com todas as épocas disponíveis. As estimativas são ordenadas por liga e época para o rank geral e por liga, divisão e época para o rank de divisão; empates de estimativa são resolvidos por `tmID`.

In [109]:
seasons = np.sort(training_data['year'].unique())
if len(seasons) < 2:
    raise ValueError('São necessárias pelo menos duas épocas para validar.')

walk_forward_rows = []
for target_year in seasons[1:]:
    historical = training_data.loc[training_data['year'] < target_year]
    current_season = training_data.loc[training_data['year'] == target_year]
    if historical.empty or current_season.empty:
        continue

    season_model = fit_ridge(
        historical[FEATURE_COLUMNS].to_numpy(dtype=np.float64),
        historical['rank'].to_numpy(dtype=np.float64),
    )
    season_predictions = predict_ridge(
        season_model,
        current_season[FEATURE_COLUMNS].to_numpy(dtype=np.float64),
    )
    season_results = current_season.copy()
    season_results['_prediction'] = season_predictions
    walk_forward_rows.append(season_results)

if not walk_forward_rows:
    raise ValueError('Não foi possível gerar previsões walk-forward.')

validation = pd.concat(walk_forward_rows, ignore_index=True)
validation_years = np.sort(validation['year'].unique())
validation_predictions = validation.pop('_prediction').to_numpy(dtype=np.float64)
validation_results = validation[['year', 'lgID', 'divID', 'tmID', 'franchID', 'rank']].copy()
validation_results = validation_results.rename(columns={'rank': 'rank_real'})
validation_results['estimativa_modelo'] = validation_predictions
validation_results = validation_results.sort_values(['year', 'tmID'])
validation_results['rank_previsto'] = (
    validation_results.groupby(['year', 'lgID'])['estimativa_modelo']
    .rank(method='first', ascending=True)
    .astype(int)
)
validation_results['rank_previsto_divisao'] = (
    validation_results.groupby(['year', 'lgID', 'divID'])['estimativa_modelo']
    .rank(method='first', ascending=True)
    .astype('Int64')
)
validation_results['acerto_rank_divisao'] = (
    validation_results['rank_previsto_divisao']
    .eq(validation_results['rank_real'])
)
validation_results['acerto_rank_divisao_ate_1'] = (
    validation_results['rank_previsto_divisao']
    .sub(validation_results['rank_real'])
    .abs()
    .le(1)
    .fillna(False)
)
mae = float(np.mean(np.abs(
    validation_predictions - validation_results['rank_real'].to_numpy(dtype=np.float64)
)))
print(f'Épocas de validação: {validation_years[0]}–{validation_years[-1]}')
print(f'Equipas validadas: {len(validation_results):,}')
print(f'MAE da estimativa contínua: {mae:.3f} posições')
validation_results = validation_results.sort_values(
    ['year', 'lgID', 'divID', 'rank_previsto_divisao', 'rank_previsto'],
    na_position='last',
).reset_index(drop=True)
RESULTS_CSV = DATA_DIR / 'team_rank_validation_results.csv'
validation_results.to_csv(RESULTS_CSV, index=False)
print(f'Tabela completa de validação guardada em: {RESULTS_CSV.resolve()}')
validation_results

Épocas de validação: 1919–2009
Equipas validadas: 1,226
MAE da estimativa contínua: 1.262 posições
Tabela completa de validação guardada em: C:\Users\cosme\UNI\AC\AC-ML-project\hockey_development_data\team_rank_validation_results.csv


,year,lgID,divID,tmID,franchID,rank_real,estimativa_modelo,rank_previsto,rank_previsto_divisao,acerto_rank_divisao,acerto_rank_divisao_ate_1
0,1919,NHL,NaN,TRS,TOR,3.0,2.046021,1,<NA>,<NA>,False
1,1919,NHL,NaN,MTL,MTL,2.0,2.363108,2,<NA>,<NA>,False
2,1919,NHL,NaN,OTS,STE,1.0,2.844088,3,<NA>,<NA>,False
3,1920,NHL,NaN,TRS,TOR,1.0,1.331154,1,<NA>,<NA>,False
4,1920,NHL,NaN,MTL,MTL,3.0,2.005002,2,<NA>,<NA>,False
...,...,...,...,...,...,...,...,...,...,...,...
1221,2009,NHL,SE,CAR,CAR,3.0,2.464553,6,1,False,False
1222,2009,NHL,SE,WAS,WAS,1.0,2.540550,7,2,False,True
1223,2009,NHL,SE,ATL,WPG,2.0,3.511852,21,3,False,True
1224,2009,NHL,SE,TBL,TBL,4.0,3.697214,26,4,True,True


## Treinar o modelo final

Depois de medir a validação, ajusta-se o modelo final com todos os exemplos disponíveis. `final_model` fica em memória para ser usado noutras células; volta a executar esta célula se reiniciar o kernel.

In [110]:
final_model = fit_ridge(
    training_data[FEATURE_COLUMNS].to_numpy(dtype=np.float64),
    training_data['rank'].to_numpy(dtype=np.float64),
)
print(f'Modelo final treinado com {len(training_data):,} exemplos.')

# Percentagem de previsões cujo rank previsto coincide exatamente com o rank real.
comparable_ranks = (
    validation_results['rank_previsto_divisao'].notna()
    & validation_results['rank_real'].notna()
)
total_predictions = int(comparable_ranks.sum())
correct_predictions = int(
    validation_results.loc[comparable_ranks, 'acerto_rank_divisao'].sum()
)
accuracy = correct_predictions / total_predictions if total_predictions else 0.0
within_one_predictions = int(
    validation_results.loc[
        comparable_ranks, 'acerto_rank_divisao_ate_1'
    ].sum()
)
within_one_accuracy = (
    within_one_predictions / total_predictions if total_predictions else 0.0
)

print(
    f'Exatidão do rank de divisão: {correct_predictions}/{total_predictions} '
    f'({accuracy:.2%})'
)
print(
    f'Exatidão com tolerância de ±1 posição: '
    f'{within_one_predictions}/{total_predictions} ({within_one_accuracy:.2%})'
)

METRICS_CSV = DATA_DIR / 'team_rank_validation_metrics.csv'
pd.DataFrame([{
    'validation_seasons': f'{validation_years[0]}-{validation_years[-1]}',
    'correct_predictions': correct_predictions,
    'total_predictions': total_predictions,
    'division_rank_exact_accuracy': accuracy,
    'division_rank_within_one_accuracy': within_one_accuracy,
    'continuous_rank_mae': mae,
}]).to_csv(METRICS_CSV, index=False)
print(f'Métricas guardadas em: {METRICS_CSV.resolve()}')

Modelo final treinado com 1,229 exemplos.
Exatidão do rank de divisão: 299/1019 (29.34%)
Exatidão com tolerância de ±1 posição: 685/1019 (67.22%)
Métricas guardadas em: C:\Users\cosme\UNI\AC\AC-ML-project\hockey_development_data\team_rank_validation_metrics.csv
